In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("Zedthecodex/KhmerTimes-Summary")

# Split train/validation
splits = dataset["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = splits["train"]
val_dataset = splits["test"]

print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")

In [ ]:
!unzip /content/mbart-khmer.zip -d /content/mbart-khmer

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model_path = "/content/mbart-khmer/mbart-khmer"
tokenizer = MBart50TokenizerFast.from_pretrained(model_path)
model = MBartForConditionalGeneration.from_pretrained(model_path)


In [ ]:
tokenizer.src_lang = "km_KH"
tokenizer.tgt_lang = "km_KH"

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

def evaluate_model(model, tokenizer, dataset, num_samples=100):
    for i in range(min(num_samples, len(dataset))):
        article = dataset[i]["Article"]
        reference = dataset[i]["Summary"]

        inputs = tokenizer(article, return_tensors="pt", truncation=True, max_length=512)
        summary_ids = model.generate(
            **inputs,
            max_length=150,
            num_beams=4,
            forced_bos_token_id=tokenizer.lang_code_to_id["km_KH"]
        )
        generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        rouge.add(prediction=generated_summary, reference=reference)

    return rouge.compute()

results = evaluate_model(model, tokenizer, val_dataset, num_samples=50)
print(results)
